In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
import lib.BTMS_model as BTMS_model

excel_path = Path("data") / "MD05E070207A1  data_power gen_single cell_Liu_20260421.xlsx"

N_cell = 320
dt_sample = 1.0

delta_t = 60

df = pd.read_excel(excel_path, sheet_name=0)
df.columns = [str(c).strip() for c in df.columns]

q_cell = pd.to_numeric(df["Qbat(W)"], errors="coerce")
q_pack = N_cell * q_cell
q_pack = q_pack.dropna().reset_index(drop=True)

time_s = np.arange(1, len(q_pack) + 1) * dt_sample

Q_peak = q_pack.max()
idx_peak = q_pack.idxmax()
t_peak = time_s[idx_peak]

Q_08_peak = 0.8 * Q_peak

window_points = int(delta_t / dt_sample)

q_avg_series = q_pack.rolling(
    window=window_points,
    min_periods=window_points
).mean()

Q_avg_dt = q_avg_series.max()
idx_end = q_avg_series.idxmax()
idx_start = idx_end - window_points + 1

Q_design = max(Q_avg_dt, Q_08_peak)

heat_load_summary = pd.DataFrame([
    {
        "item": "Q_peak",
        "value_W": Q_peak,
        "time_s": t_peak
    },
    {
        "item": f"Q_avg_{delta_t}s",
        "value_W": Q_avg_dt,
        "time_s": f"{time_s[idx_start]:.0f}-{time_s[idx_end]:.0f}"
    },
    {
        "item": "0.8Q_peak",
        "value_W": Q_08_peak,
        "time_s": "-"
    },
    {
        "item": "Q_design",
        "value_W": Q_design,
        "time_s": "-"
    }
])

heat_load_summary["value_W"] = heat_load_summary["value_W"].round(3)

display(heat_load_summary)

print(f"delta_t = {delta_t} s")
print(f"Q_peak = {Q_peak:.3f} W")
print(f"Q_avg_dt = {Q_avg_dt:.3f} W")
print(f"0.8Q_peak = {Q_08_peak:.3f} W")
print(f"Q_design = {Q_design:.3f} W")


,item,value_W,time_s
0,Q_peak,1329.005,1920.0
1,Q_avg_60s,1276.329,1861-1920
2,0.8Q_peak,1063.204,-
3,Q_design,1276.329,-


delta_t = 60 s
Q_peak = 1329.005 W
Q_avg_dt = 1276.329 W
0.8Q_peak = 1063.204 W
Q_design = 1276.329 W


In [8]:
# ============================================================
# Cell 2 - Step 2: Calculate the required coolant mass flow rate
# ============================================================

cp_f = 4180.0              # J/(kg·K), water

# 可修改参数
DeltaT_f_allow = 5.0       # K, allowable coolant temperature rise

# 冷却液总质量流量
m_dot_f = Q_design / (cp_f * DeltaT_f_allow)

flow_rate_summary = pd.DataFrame([
    {
        "Q_design_W": Q_design,
        "cp_f_J_kgK": cp_f,
        "DeltaT_f_allow_K": DeltaT_f_allow,
        "m_dot_f_kg_s": m_dot_f
    }
])

flow_rate_summary = flow_rate_summary.round(4)

display(flow_rate_summary)

print("Step 2 results")
print("-" * 60)
print(f"Q_design = {Q_design:.3f} W")
print(f"DeltaT_f_allow = {DeltaT_f_allow:.1f} K")
print(f"cp_f = {cp_f:.1f} J/(kg·K)")
print(f"m_dot_f = {m_dot_f:.5f} kg/s")

,Q_design_W,cp_f_J_kgK,DeltaT_f_allow_K,m_dot_f_kg_s
0,1276.3287,4180.0,5.0,0.0611


Step 2 results
------------------------------------------------------------
Q_design = 1276.329 W
DeltaT_f_allow = 5.0 K
cp_f = 4180.0 J/(kg·K)
m_dot_f = 0.06107 kg/s


In [9]:
T_b_lim = 45.0          # °C, battery temperature limit
T_f_in = 25.0           # °C, coolant inlet temperature

# 冷却液总热容量率
C_f = m_dot_f * cp_f

# 整块冷板的最大可能换热量
Q_max = C_f * (T_b_lim - T_f_in)

# 整块冷板所需换热有效度
epsilon = Q_design / Q_max

design_condition_summary = pd.DataFrame([
    {
        "Q_design_W": Q_design,
        "T_b_lim_C": T_b_lim,
        "T_f_in_C": T_f_in,
        "C_f_W_K": C_f,
        "Q_max_W": Q_max,
        "epsilon": epsilon
    }
])

display(design_condition_summary.round(4))

print("Step 3 results")
print("-" * 60)
print(f"Q_design = {Q_design:.3f} W")
print(f"T_b_lim = {T_b_lim:.1f} °C")
print(f"T_f_in = {T_f_in:.1f} °C")
print(f"C_f = {C_f:.3f} W/K")
print(f"Q_max = {Q_max:.3f} W")
print(f"epsilon = {epsilon:.6f}")

,Q_design_W,T_b_lim_C,T_f_in_C,C_f_W_K,Q_max_W,epsilon
0,1276.3287,45.0,25.0,255.2657,5105.315,0.25


Step 3 results
------------------------------------------------------------
Q_design = 1276.329 W
T_b_lim = 45.0 °C
T_f_in = 25.0 °C
C_f = 255.266 W/K
Q_max = 5105.315 W
epsilon = 0.250000


In [10]:
# ============================================================
# Cell 4 - Step 4: Calculate the required total UA
# ============================================================

# 在电池侧温度近似保持为 T_b_lim 的恒温壁面假设下
NTU = -np.log(1.0 - epsilon)

UA_design = C_f * NTU

UA_design_summary = pd.DataFrame([
    {
        "Q_design_W": Q_design,
        "Q_max_W": Q_max,
        "epsilon": epsilon,
        "NTU": NTU,
        "C_f_W_K": C_f,
        "UA_design_W_K": UA_design
    }
])

display(UA_design_summary.round(4))

print("Step 4 results")
print("-" * 60)
print(f"Q_design = {Q_design:.3f} W")
print(f"Q_max = {Q_max:.3f} W")
print(f"epsilon = {epsilon:.6f}")
print(f"NTU = {NTU:.6f}")
print(f"C_f = {C_f:.3f} W/K")
print(f"UA_design = {UA_design:.3f} W/K")

,Q_design_W,Q_max_W,epsilon,NTU,C_f_W_K,UA_design_W_K
0,1276.3287,5105.315,0.25,0.2877,255.2657,73.4354


Step 4 results
------------------------------------------------------------
Q_design = 1276.329 W
Q_max = 5105.315 W
epsilon = 0.250000
NTU = 0.287682
C_f = 255.266 W/K
UA_design = 73.435 W/K


In [11]:
# ============================================================
# Cell 5 - Rectangular parallel-channel sizing
# ============================================================

rho_f = 997.0
mu_f = 0.00089
k_f = 0.606

k_plate = 200.0
t_plate = 0.002

W_module = 0.35550
L_module = 0.38823

N_ch = 20
W_ch = 0.016
t_rib = 0.002
t_side = 0.002

W_pitch_est = W_module / N_ch
W_plate = N_ch * W_ch + (N_ch - 1) * t_rib + 2.0 * t_side
width_margin = W_plate - W_module


m_dot_ch = m_dot_f / N_ch

H_ch_values_mm = np.round(
    np.arange(1.0, 6.0 + 0.05, 0.05),
    2
)

scan_results = []

for H_ch_mm in H_ch_values_mm:

    H_ch_i = H_ch_mm / 1000.0

    A_ch_i = W_ch * H_ch_i
    P_wetted_i = 2.0 * (W_ch + H_ch_i)
    D_h_i = 4.0 * A_ch_i / P_wetted_i

    u_ch_i = m_dot_ch / (rho_f * A_ch_i)
    Re_i = rho_f * u_ch_i * D_h_i / mu_f

    if Re_i >= 2300.0:
        continue

    Nu_i = BTMS_model.liquid_nusselt_number(W_ch,H_ch_i)

    h_f_i = Nu_i * k_f / D_h_i

    U_i = 1.0 / (
        1.0 / h_f_i
        + t_plate / k_plate
    )

    A_required_i = UA_design / U_i
    L_ch_i = A_required_i / (N_ch * W_ch)

    scan_results.append({
        "H_ch_mm": H_ch_mm,
        "D_h_mm": D_h_i * 1000.0,
        "u_ch_m_s": u_ch_i,
        "Re": Re_i,
        "Nu": Nu_i,
        "h_f_W_m2K": h_f_i,
        "U_W_m2K": U_i,
        "A_required_m2": A_required_i,
        "L_ch_mm": L_ch_i * 1000.0,
        "length_margin_mm": (L_ch_i - L_module) * 1000.0
    })

scan_table = pd.DataFrame(scan_results)

feasible_table = scan_table[
    scan_table["L_ch_mm"] >= L_module * 1000.0
].copy()


best_design = feasible_table.loc[
    feasible_table["length_margin_mm"].idxmin()
]

H_ch = best_design["H_ch_mm"] / 1000.0
D_h = best_design["D_h_mm"] / 1000.0
u_ch = best_design["u_ch_m_s"]
Re = best_design["Re"]
Nu = best_design["Nu"]
h_f = best_design["h_f_W_m2K"]
U = best_design["U_W_m2K"]
A_required = best_design["A_required_m2"]
L_ch = best_design["L_ch_mm"] / 1000.0

geometry_summary = pd.DataFrame([{
    "Park_width_mm": W_module * 1000.0,
    "Park_length_mm": L_module * 1000.0,
    "Average_width_per_channel_mm": W_pitch_est * 1000.0,
    "Channel_number": N_ch,
    "Channel_width_mm": W_ch * 1000.0,
    "Channel_height_mm": H_ch * 1000.0,
    "Rib_thickness_mm": t_rib * 1000.0,
    "Side_wall_thickness_mm": t_side * 1000.0,
    "Cold_plate_width_mm": W_plate * 1000.0,
    "Width_margin_mm": width_margin * 1000.0,
    "Hydraulic_diameter_mm": D_h * 1000.0,
    "Channel_velocity_m_s": u_ch,
    "Re": Re,
    "Nu": Nu,
    "h_f_W_m2K": h_f,
    "U_W_m2K": U,
    "UA_design_W_K": UA_design,
    "Required_area_m2": A_required,
    "Channel_length_mm": L_ch * 1000.0,
    "Length_margin_mm": (L_ch - L_module) * 1000.0
}])

display(geometry_summary.round(4))

print("Baseline rectangular-channel design")
print("-" * 50)
print(f"Average available width : {W_pitch_est * 1000:.3f} mm")
print(f"Channel width          : {W_ch * 1000:.2f} mm")
print(f"Channel height         : {H_ch * 1000:.2f} mm")
print(f"Cold-plate width       : {W_plate * 1000:.2f} mm")
print(f"Channel length         : {L_ch * 1000:.2f} mm")
print(f"Width margin           : {width_margin * 1000:.2f} mm")
print(f"Length margin          : {(L_ch - L_module) * 1000:.2f} mm")

,Park_width_mm,Park_length_mm,Average_width_per_channel_mm,Channel_number,Channel_width_mm,Channel_height_mm,Rib_thickness_mm,Side_wall_thickness_mm,Cold_plate_width_mm,Width_margin_mm,Hydraulic_diameter_mm,Channel_velocity_m_s,Re,Nu,h_f_W_m2K,U_W_m2K,UA_design_W_K,Required_area_m2,Channel_length_mm,Length_margin_mm
0,355.5,388.23,17.775,20,16.0,3.5,2.0,2.0,362.0,6.5,5.7436,0.0547,351.8776,5.5778,588.5117,585.0685,73.4354,0.1255,392.2371,4.0071


Baseline rectangular-channel design
--------------------------------------------------
Average available width : 17.775 mm
Channel width          : 16.00 mm
Channel height         : 3.50 mm
Cold-plate width       : 362.00 mm
Channel length         : 392.24 mm
Width margin           : 6.50 mm
Length margin          : 4.01 mm


In [12]:
# ============================================================
# Cell 6 - Pressure drop and pumping power
# ============================================================

t_mission = 1920.0
eta_pump = 0.35

power_args = {
    "fluid_cool": "water",
    "rho_cool": rho_f,
    "mu_cool": mu_f,
    "W_channel": W_ch,
    "H_channel": H_ch,
    "L_channel": L_ch,
    "m_dot_total": m_dot_f,
    "num_channel": N_ch,
    "pump_efficiency": eta_pump,
    "operation_time": t_mission,
    "K_minor": 0.0,
}

power_results = BTMS_model.cal_btms_aux_power(
    power_args
)

Re = power_results["Re"]
f = power_results["friction_factor"]
u_ch = power_results["u_cool_in_m_s"]
DeltaP = power_results["delta_p_Pa"]
V_dot_f = power_results["V_dot_total_m3_s"]
P_pump = power_results["P_aux_W"]
E_pump_J = power_results["E_aux_J"]

P_hydraulic = DeltaP * V_dot_f
E_pump_Wh = E_pump_J / 3600.0

if Re < 2300.0:
    flow_regime = "laminar"
elif Re >= 4000.0:
    flow_regime = "turbulent"
else:
    flow_regime = "transition"

pump_summary = pd.DataFrame([{
    "Flow_regime": flow_regime,
    "Re": Re,
    "Friction_factor": f,
    "Pressure_drop_Pa": DeltaP,
    "Volume_flow_rate_L_min": V_dot_f * 60000.0,
    "Hydraulic_power_W": P_hydraulic,
    "Pump_efficiency": eta_pump,
    "Pump_power_W": P_pump,
    "Mission_time_s": t_mission,
    "Pump_energy_Wh": E_pump_Wh
}])

display(pump_summary.round(6))

print("Pressure drop and pumping power")
print("-" * 50)
print(f"Flow regime       : {flow_regime}")
print(f"Re                : {Re:.2f}")
print(f"Friction factor   : {f:.6f}")
print(f"Pressure drop     : {DeltaP:.4f} Pa")
print(f"Volume flow rate  : {V_dot_f * 60000.0:.4f} L/min")
print(f"Hydraulic power   : {P_hydraulic:.6f} W")
print(f"Pump efficiency   : {eta_pump:.2f}")
print(f"Pump power        : {P_pump:.6f} W")
print(f"Pump energy       : {E_pump_Wh:.6f} Wh")

,Flow_regime,Re,Friction_factor,Pressure_drop_Pa,Volume_flow_rate_L_min,Hydraulic_power_W,Pump_efficiency,Pump_power_W,Mission_time_s,Pump_energy_Wh
0,laminar,351.877619,0.181881,18.519313,3.675127,0.001134,0.35,0.003241,1920.0,0.001729


Pressure drop and pumping power
--------------------------------------------------
Flow regime       : laminar
Re                : 351.88
Friction factor   : 0.181881
Pressure drop     : 18.5193 Pa
Volume flow rate  : 3.6751 L/min
Hydraulic power   : 0.001134 W
Pump efficiency   : 0.35
Pump power        : 0.003241 W
Pump energy       : 0.001729 Wh


In [13]:
# ============================================================
# Cell 7 - Liquid-cooling BTMS mass
# ============================================================

rho_plate = 2700.0       # kg/m3, aluminium
D_bat = 0.018            # m, battery-cell diameter
N_c = 20                 # number of cell columns

# Equivalent pitch used to reproduce the designed cold-plate width.
S_T = (
    W_plate - D_bat
) / (N_c - 1)

# Set to zero when the actual pump and pipe masses are unavailable.
m_pump = 0.0
m_pipe = 0.0

mass_args = {
    "fluid_cool": "water",
    "rho_cool": rho_f,
    "rho_plate": rho_plate,

    "S_T": S_T,
    "D_bat": D_bat,
    "N_c": N_c,

    "W_channel": W_ch,
    "H_channel": H_ch,
    "L_channel": L_ch,
    "num_channel": N_ch,

    "plate_extra_height": t_plate,

    "m_pump": m_pump,
    "m_pipe": m_pipe,

    "return_components": True,
}

mass_results = BTMS_model.cal_btms_mass(
    mass_args
)

m_plate = mass_results["m_plate_kg"]
m_coolant = mass_results["m_coolant_kg"]
m_BTMS = mass_results["m_BTMS_kg"]

mass_summary = pd.DataFrame([{
    "Cold_plate_width_mm": W_plate * 1000.0,
    "Cold_plate_thickness_mm": (
        H_ch + t_plate
    ) * 1000.0,
    "Channel_number": N_ch,
    "Channel_width_mm": W_ch * 1000.0,
    "Channel_height_mm": H_ch * 1000.0,
    "Channel_length_mm": L_ch * 1000.0,
    "Plate_mass_kg": m_plate,
    "Coolant_mass_kg": m_coolant,
    "Pump_mass_kg": mass_results["m_pump_kg"],
    "Pipe_mass_kg": mass_results["m_pipe_kg"],
    "Total_BTMS_mass_kg": m_BTMS,
}])

display(mass_summary.round(6))

print("Liquid-cooling BTMS mass")
print("-" * 50)
print(f"Cold-plate width     : {W_plate * 1000.0:.2f} mm")
print(f"Cold-plate thickness : {(H_ch + t_plate) * 1000.0:.2f} mm")
print(f"Cold-plate mass      : {m_plate:.6f} kg")
print(f"Coolant mass         : {m_coolant:.6f} kg")
print(f"Pump mass            : {mass_results['m_pump_kg']:.6f} kg")
print(f"Pipe mass            : {mass_results['m_pipe_kg']:.6f} kg")
print(f"Total BTMS mass      : {m_BTMS:.6f} kg")

,Cold_plate_width_mm,Cold_plate_thickness_mm,Channel_number,Channel_width_mm,Channel_height_mm,Channel_length_mm,Plate_mass_kg,Coolant_mass_kg,Pump_mass_kg,Pipe_mass_kg,Total_BTMS_mass_kg
0,362.0,5.5,20,16.0,3.5,392.237054,0.922424,0.437988,0.0,0.0,1.360411


Liquid-cooling BTMS mass
--------------------------------------------------
Cold-plate width     : 362.00 mm
Cold-plate thickness : 5.50 mm
Cold-plate mass      : 0.922424 kg
Coolant mass         : 0.437988 kg
Pump mass            : 0.000000 kg
Pipe mass            : 0.000000 kg
Total BTMS mass      : 1.360411 kg
